# 01 — Baseline: faster-whisper large-v3 на одном RU треке

Цель: получить первые цифры — RTF, language detection, hallucination rate, WER против найденной лирики.

Запускать на **Colab T4** (Runtime → Change runtime type → GPU → T4).

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU не подключен — Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/RezPoint/unbake-test.git
%cd unbake-test
!pip install -q faster-whisper jiwer pydantic requests

In [ ]:
!python notebooks/download_dataset.py
!ls data/raw/ru/

In [ ]:
# Baseline на одном треке
from src.baseline import transcribe

TRACK = 'data/raw/ru/Pharaoh - Дико, например.m4a'
transcript, telemetry = transcribe(TRACK, language='ru', model_size='large-v3', device='cuda', compute_type='float16')
print(telemetry)

In [ ]:
# Первые 30 слов с таймкодами
for w in transcript.words[:30]:
    print(f'{w.start:6.2f} → {w.end:6.2f}  {w.text}  (p={w.confidence:.2f})')

In [ ]:
# Сохраняем result для last в репо нельзя (чтение-только) → пишем в Colab локально и качаем
import json
with open('/content/baseline_pharaoh_ru.json', 'w', encoding='utf-8') as f:
    f.write(transcript.model_dump_json(indent=2))
with open('/content/baseline_pharaoh_ru.telemetry.json', 'w', encoding='utf-8') as f:
    json.dump(telemetry, f, indent=2)
print('saved → /content/baseline_pharaoh_ru.json')
print('saved → /content/baseline_pharaoh_ru.telemetry.json')

## Что смотреть в результате

**telemetry:**
- `rtf` — real-time factor. На T4 для large-v3 ожидаем 0.10-0.20 (т.е. 3-минутный трек обрабатывается за 18-36 сек).
- `lang_prob` — уверенность в языке. < 0.9 на RU-треке = тревога.
- `audio_s` / `infer_s` — для cost-расчёта.

**По словам:**
- Глазами проверить, попадают ли границы слов в реальные позиции (можно сверить с любым плеером).
- `confidence < 0.4` — кандидат на отбраковку или fallback.

**Cost экстраполяция:** RTF на T4 → пересчёт на A10G (T4 ≈ 8 TFLOPS fp16, A10G ≈ 31 TFLOPS fp16, т.е. ~3.5× быстрее). Цены Runpod: A10G ≈ $0.34/ч → cost = (audio_s × rtf_a10g) / 3600 × 0.34.